# Physics-Aware Domain Adaptation and Lightweight Uncertainty-Guided Cancer Screening

**PA-LUCDNet: complete PyTorch implementation**

This notebook implements the six-stage pipeline: paired PACT/URCT preprocessing, physics-guided adaptive feature learning (Beer–Lambert-inspired consistency + MMD), MobileNetV3-Small gated fusion, Monte Carlo-dropout classification, temperature scaling/selective referral, Grad-CAM++, patient-level evaluation, and deployment optimization.

**Dataset facts:** the linked Dryad release describes 161 breast tumors and 322 MATLAB `.mat` files (PACT + URCT). The actual variable names, patient-to-file pairing, pathology labels, and acquisition metadata must be inspected from the downloaded files. This notebook deliberately does not invent labels or patient identities.

## 0. Runtime and reproducibility

Run in Google Colab or a GPU-enabled Jupyter environment. The first cell installs dependencies. Set `DATA_ROOT` to the extracted Dryad `DATA` directory. A patient-level manifest is required because labels and pairing must come from the dataset documentation/verified metadata.

In [ ]:
%pip -q install numpy scipy pandas scikit-learn matplotlib seaborn tqdm openpyxl

In [ ]:
import os, re, json, math, random, time, copy, warnings, glob
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    brier_score_loss, classification_report)
from sklearn.calibration import calibration_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from scipy.ndimage import median_filter
from skimage import exposure, transform
warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_ROOT = Path("/content/DATA")  # CHANGE to extracted Dryad DATA directory
WORK_DIR = Path("Gowshick sir")
WORK_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 40
LR = 2e-4
NUM_WORKERS = 2

## 1. Dataset inspection and manifest construction

Extract `DATA.zip` from Dryad, then point `DATA_ROOT` to the folder containing the `.mat` files. First inspect the file names and MATLAB variable keys. Use the output to determine the exact PACT/URCT pairing and the correct pathology label mapping.

Create `manifest.csv` with these required columns:

- `patient_id`: stable patient/tumor identifier; both modalities for one tumor must share it.
- `pact_path`, `urct_path`: paths to paired `.mat` files (absolute or relative to `DATA_ROOT`).
- `label`: `0` benign, `1` malignant, based on verified pathology ground truth.
- `domain`: acquisition group identifier, if available; otherwise use `default`.
- Optional physics columns: `i0`, `mu_a`, `distance`, `fluence` (use actual calibrated metadata only).

Do not infer pathology labels from filenames unless the dataset documentation explicitly defines that convention.

In [ ]:
mat_files = sorted(DATA_ROOT.rglob("*.mat")) if DATA_ROOT.exists() else []
print("MAT files found:", len(mat_files))
for p in mat_files[:10]:
    try:
        m = sio.loadmat(p)
        keys = [k for k in m.keys() if not k.startswith("__")]
        shapes = {k: np.asarray(m[k]).shape for k in keys}
        print("\\n", p.name, shapes)
    except Exception as e:
        print(p.name, "READ ERROR:", e)

In [ ]:
# Optional: create a starter manifest from a verified pairing table.
# This only lists files; it does NOT guess patient IDs, modality, or labels.
if mat_files:
    starter = pd.DataFrame({"mat_file": [str(p) for p in mat_files]})
    starter.to_csv(WORK_DIR / "mat_file_inventory.csv", index=False)
    display(starter.head())

In [ ]:
MANIFEST_PATH = DATA_ROOT / "manifest.csv"  # create this file after verifying pairing/labels
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Create {MANIFEST_PATH} with patient_id,pact_path,urct_path,label,domain columns. "
        "Use the inspection output and verified Dryad metadata; do not guess labels."
    )
df = pd.read_csv(MANIFEST_PATH)
required = {"patient_id","pact_path","urct_path","label"}
missing = required - set(df.columns)
if missing: raise ValueError(f"Manifest missing columns: {sorted(missing)}")
if not set(df.label.dropna().unique()).issubset({0,1}):
    raise ValueError("Encode verified labels as 0=benign and 1=malignant.")
if df.patient_id.duplicated().any():
    raise ValueError("Expected one paired PACT/URCT row per patient/tumor. Resolve duplicate rows.")
df["domain"] = df["domain"].fillna("default").astype(str) if "domain" in df else "default"
for c in ["pact_path","urct_path"]:
    df[c] = df[c].apply(lambda x: str((DATA_ROOT / x).resolve()) if not Path(str(x)).is_absolute() else str(Path(str(x))))
    absent = [x for x in df[c] if not Path(x).exists()]
    if absent: raise FileNotFoundError(f"{c}: missing file, example: {absent[0]}")
print(df.shape, df.label.value_counts().to_dict(), "domains:", df.domain.nunique())
display(df.head())

## 2. MATLAB loading, preprocessing, and paired patient-level splits

The loader selects a numeric image-like array from each `.mat` file. If a file contains multiple candidate arrays, set `PACT_KEY` and `URCT_KEY` explicitly after inspecting the variable-key output above. Median filtering suppresses speckle/noise; CLAHE improves local contrast. Resize and normalization are applied consistently.

In [ ]:
PACT_KEY = None  # e.g. "PACT_image"; set after inspecting .mat keys
URCT_KEY = None  # e.g. "URCT_image"

def load_mat_image(path, key=None):
    obj = sio.loadmat(path)
    if key is not None:
        if key not in obj: raise KeyError(f"{key} not found in {path}; keys={list(obj.keys())}")
        arr = np.asarray(obj[key])
    else:
        candidates = []
        for k,v in obj.items():
            if k.startswith("__"): continue
            a = np.asarray(v)
            if a.dtype.kind in "fiu" and a.ndim >= 2 and a.size > 256:
                candidates.append((a.size, k, a))
        if not candidates:
            raise ValueError(f"No numeric image-like array found in {path}")
        candidates.sort(reverse=True, key=lambda z:z[0])
        arr = candidates[0][2]
        if len(candidates) > 1:
            print(f"Using largest array '{candidates[0][1]}' from {Path(path).name}; verify key selection.")
    arr = np.squeeze(arr)
    while arr.ndim > 2:
        arr = arr[..., 0]
    if arr.ndim != 2: raise ValueError(f"Expected 2D image, got {arr.shape} in {path}")
    arr = np.nan_to_num(arr.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return arr

def preprocess_image(arr, size=IMG_SIZE):
    arr = median_filter(arr, size=3)
    lo, hi = np.percentile(arr, [1, 99])
    arr = np.clip((arr-lo)/(hi-lo+1e-8), 0, 1)
    arr = exposure.equalize_adapthist(arr, clip_limit=0.02)
    arr = transform.resize(arr, (size,size), order=1, preserve_range=True, anti_aliasing=True)
    arr = np.clip(arr, 0, 1).astype(np.float32)
    return arr

# Split by patient/tumor, stratified by label. No patient appears in more than one partition.
train_df, hold_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df.label)
val_df, test_df = train_test_split(hold_df, test_size=0.50, random_state=SEED, stratify=hold_df.label)
print("Patients:", len(train_df), len(val_df), len(test_df))
assert set(train_df.patient_id).isdisjoint(val_df.patient_id)
assert set(train_df.patient_id).isdisjoint(test_df.patient_id)
assert set(val_df.patient_id).isdisjoint(test_df.patient_id)

## 3. Dataset and loaders

In [ ]:
class PairedPACTDataset(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        r = self.frame.iloc[idx]
        pact = preprocess_image(load_mat_image(r.pact_path, PACT_KEY))
        urct = preprocess_image(load_mat_image(r.urct_path, URCT_KEY))
        if self.augment and random.random() < 0.5:
            pact, urct = np.fliplr(pact).copy(), np.fliplr(urct).copy()
        if self.augment and random.random() < 0.25:
            scale = random.uniform(0.9,1.1)
            pact = np.clip(pact*scale,0,1); urct = np.clip(urct*scale,0,1)
        x1 = torch.from_numpy(pact[None]).float().repeat(3,1,1)
        x2 = torch.from_numpy(urct[None]).float().repeat(3,1,1)
        y = torch.tensor(float(r.label), dtype=torch.float32)
        domain = str(r.domain)
        # Optional calibrated optical parameters. Missing values use neutral defaults.
        getv = lambda c,d: float(r[c]) if c in self.frame.columns and pd.notna(r[c]) else d
        physics = torch.tensor([getv("i0",1.0), getv("mu_a",0.0),
                               getv("distance",1.0), getv("fluence",1.0)], dtype=torch.float32)
        return x1, x2, y, domain, physics, str(r.patient_id)

def collate_batch(batch):
    a,b,y,d,p,ids = zip(*batch)
    return torch.stack(a),torch.stack(b),torch.stack(y),list(d),torch.stack(p),list(ids)

train_loader = DataLoader(PairedPACTDataset(train_df,True), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_batch)
val_loader = DataLoader(PairedPACTDataset(val_df), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_batch)
test_loader = DataLoader(PairedPACTDataset(test_df), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, collate_fn=collate_batch)

## 4. PAFL: Beer–Lambert-inspired physics constraint + MMD domain adaptation

Beer–Lambert attenuation is represented as `I = I0 * exp(-mu_a * distance)`. A physics consistency loss compares a normalized image-intensity proxy with the attenuation predicted from supplied optical parameters. **This is only physically calibrated when `i0`, `mu_a`, and `distance` are genuine acquisition measurements with consistent units.** If those measurements are unavailable, the default physics loss is disabled rather than pretending image intensity is calibrated optical fluence.

MMD aligns latent feature distributions across acquisition domains. Domain labels are used only to form batches/groups; the diagnostic target remains pathology.

In [ ]:
def rbf_mmd(x, y, sigmas=(0.1,1,10)):
    if x.size(0) < 2 or y.size(0) < 2: return x.new_tensor(0.)
    xx = x.unsqueeze(1)-x.unsqueeze(0)
    yy = y.unsqueeze(1)-y.unsqueeze(0)
    xy = x.unsqueeze(1)-y.unsqueeze(0)
    kxx = sum(torch.exp(-xx.pow(2).mean(-1)/(2*s)) for s in sigmas)
    kyy = sum(torch.exp(-yy.pow(2).mean(-1)/(2*s)) for s in sigmas)
    kxy = sum(torch.exp(-xy.pow(2).mean(-1)/(2*s)) for s in sigmas)
    return kxx.mean()+kyy.mean()-2*kxy.mean()

def physics_beer_lambert_loss(pact, physics):
    # physics columns: I0, mu_a, distance, fluence. Require positive measured I0 and mu_a.
    i0, mu_a, distance, fluence = [physics[:,i] for i in range(4)]
    valid = (i0 > 0) & (mu_a > 0) & (distance > 0) & (fluence > 0)
    if not valid.any(): return pact.new_tensor(0.)
    pred = i0[valid] * torch.exp(-mu_a[valid]*distance[valid])
    target = fluence[valid]
    pred = pred/(pred.detach().abs().mean()+1e-8)
    target = target/(target.detach().abs().mean()+1e-8)
    return F.smooth_l1_loss(pred, target)

## 5. PA-LUCDNet architecture: MobileNetV3-Small dual encoder, gated fusion, uncertainty head

In [ ]:
class MobileEncoder(nn.Module):
    def __init__(self, pretrained=True, out_dim=256):
        super().__init__()
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        net = models.mobilenet_v3_small(weights=weights)
        self.features = net.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(576,out_dim), nn.Hardswish())
    def forward(self,x):
        return self.proj(self.pool(self.features(x)))

class PA_LUCDNet(nn.Module):
    def __init__(self, pretrained=True, dropout=0.35, feat_dim=256):
        super().__init__()
        self.pact_encoder = MobileEncoder(pretrained,feat_dim)
        self.urct_encoder = MobileEncoder(pretrained,feat_dim)
        self.gate = nn.Sequential(nn.Linear(feat_dim*2,feat_dim),nn.Sigmoid())
        self.fusion = nn.Sequential(nn.Linear(feat_dim*2,feat_dim),nn.BatchNorm1d(feat_dim),nn.Hardswish())
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(feat_dim,1)
    def forward(self,pact,urct,return_features=False):
        fp = self.pact_encoder(pact)
        fu = self.urct_encoder(urct)
        g = self.gate(torch.cat([fp,fu],1))
        fused = self.fusion(torch.cat([g*fp,(1-g)*fu],1))
        logits = self.classifier(self.dropout(fused)).squeeze(1)
        if return_features: return logits,fused,fp,fu
        return logits

model = PA_LUCDNet(pretrained=True).to(DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 6. Training: classification + MMD + optional physics consistency

In [ ]:
LAMBDA_MMD = 0.05
LAMBDA_PHYSICS = 0.0  # set >0 only after supplying calibrated physics metadata
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=5, factor=0.5)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def run_epoch(loader, training=False):
    model.train(training)
    losses=[]; ys=[]; probs=[]
    for pact,urct,y,domains,physics,_ in tqdm(loader, leave=False):
        pact,urct,y,physics = pact.to(DEVICE),urct.to(DEVICE),y.to(DEVICE),physics.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=="cuda")):
                logits,fused,fp,fu = model(pact,urct,True)
                cls = F.binary_cross_entropy_with_logits(logits,y)
                mmd = logits.new_tensor(0.)
                unique = sorted(set(domains))
                for i in range(len(unique)):
                    for j in range(i+1,len(unique)):
                        xi = fused[torch.tensor([d==unique[i] for d in domains],device=DEVICE)]
                        xj = fused[torch.tensor([d==unique[j] for d in domains],device=DEVICE)]
                        if len(xi)>1 and len(xj)>1: mmd = mmd + rbf_mmd(xi,xj)
                phys = physics_beer_lambert_loss(pact.mean((1,2,3)),physics)
                loss = cls + LAMBDA_MMD*mmd + LAMBDA_PHYSICS*phys
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
        losses.append(loss.item())
        ys.extend(y.detach().cpu().numpy().tolist())
        probs.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
    pred = np.asarray(probs)>=0.5
    return float(np.mean(losses)), accuracy_score(ys,pred), (roc_auc_score(ys,probs) if len(set(ys))>1 else np.nan)

history=[]
best_auc=-np.inf
for epoch in range(1,EPOCHS+1):
    tr=run_epoch(train_loader,True); va=run_epoch(val_loader,False)
    scheduler.step(va[2] if np.isfinite(va[2]) else va[1])
    history.append([epoch,*tr,*va])
    print(f"Epoch {epoch:03d} | train loss {tr[0]:.4f} acc {tr[1]:.4f} AUC {tr[2]:.4f} | val loss {va[0]:.4f} acc {va[1]:.4f} AUC {va[2]:.4f}")
    score=va[2] if np.isfinite(va[2]) else va[1]
    if score>best_auc:
        best_auc=score
        torch.save(model.state_dict(),WORK_DIR/"pa_lucdnet_best.pt")
hist=pd.DataFrame(history,columns=["epoch","train_loss","train_acc","train_auc","val_loss","val_acc","val_auc"])
display(hist.tail())

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(hist.epoch,hist.train_loss,label="Train"); ax[0].plot(hist.epoch,hist.val_loss,label="Validation")
ax[0].set_title("Training and validation loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(hist.epoch,hist.train_auc,label="Train AUROC"); ax[1].plot(hist.epoch,hist.val_auc,label="Validation AUROC")
ax[1].set_title("AUROC"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. Monte Carlo Dropout, temperature scaling, and calibrated probabilities

In [ ]:
model.load_state_dict(torch.load(WORK_DIR/"pa_lucdnet_best.pt",map_location=DEVICE))
def mc_predict(loader, passes=30):
    model.train()  # activate dropout at inference; BatchNorm statistics are frozen below
    for m in model.modules():
        if isinstance(m,nn.BatchNorm2d) or isinstance(m,nn.BatchNorm1d): m.eval()
    all_mc=[]; ys=[]; ids=[]
    with torch.no_grad():
        for pact,urct,y,domains,physics,pids in loader:
            pact,urct=pact.to(DEVICE),urct.to(DEVICE)
            samples=[]
            for _ in range(passes):
                samples.append(torch.sigmoid(model(pact,urct)).cpu().numpy())
            all_mc.append(np.stack(samples,axis=0))
            ys.extend(y.numpy().tolist()); ids.extend(pids)
    mc=np.concatenate(all_mc,axis=1)
    return mc,np.asarray(ys),np.asarray(ids)

val_mc,y_val,val_ids=mc_predict(val_loader,30)
val_prob=val_mc.mean(0)
val_unc=val_mc.var(0)

In [ ]:
# Fit scalar temperature on validation logits via a 1D grid search.
def mc_logits_for_temperature(loader,passes=10):
    model.train()
    for m in model.modules():
        if isinstance(m,(nn.BatchNorm1d,nn.BatchNorm2d)): m.eval()
    logits_all=[]; labels=[]
    with torch.no_grad():
        for p,u,y,d,ph,ids in loader:
            p,u=p.to(DEVICE),u.to(DEVICE)
            ls=[model(p,u).cpu().numpy() for _ in range(passes)]
            logits_all.append(np.mean(ls,axis=0)); labels.extend(y.numpy())
    return np.concatenate(logits_all),np.asarray(labels)
val_logits,yv=mc_logits_for_temperature(val_loader)
temps=np.linspace(0.5,3.0,101)
best_temp=min(temps,key=lambda t:F.binary_cross_entropy_with_logits(
    torch.tensor(val_logits/t),torch.tensor(yv,dtype=torch.float32)).item())
print("Selected temperature:",best_temp)
val_cal=1/(1+np.exp(-val_logits/best_temp))

## 8. Selective classification and three-level screening/referral policy

Thresholds are selected from validation data only. A high uncertainty or intermediate calibrated malignancy probability triggers specialist review. The policy is configurable and must be clinically validated before any clinical use.

In [ ]:
LOW_RISK_MAX = 0.20
HIGH_RISK_MIN = 0.80
UNCERTAINTY_MAX = 0.02  # variance of MC probabilities; tune using validation set only

def screening_decision(prob, uncertainty):
    if uncertainty > UNCERTAINTY_MAX or LOW_RISK_MAX < prob < HIGH_RISK_MIN:
        return "specialist-review-required"
    if prob >= HIGH_RISK_MIN: return "elevated-risk"
    return "low-risk"

val_decisions=[screening_decision(p,u) for p,u in zip(val_cal,val_unc)]
print(pd.Series(val_decisions).value_counts())

## 9. Test evaluation: diagnostic, discrimination, uncertainty, referral, coverage, selective risk

In [ ]:
test_mc,y_test,test_ids=mc_predict(test_loader,50)
test_prob_raw=test_mc.mean(0)
test_unc=test_mc.var(0)
test_prob=1/(1+np.exp(-((np.log(np.clip(test_prob_raw,1e-6,1-1e-6)/(1-np.clip(test_prob_raw,1e-6,1-1e-6)))/best_temp))))
test_pred=(test_prob>=0.5).astype(int)
test_decisions=[screening_decision(p,u) for p,u in zip(test_prob,test_unc)]
accepted=np.array([d!="specialist-review-required" for d in test_decisions])
selective_risk=(1-accuracy_score(y_test[accepted],test_pred[accepted])) if accepted.any() else np.nan
metrics={
 "Accuracy":accuracy_score(y_test,test_pred),
 "Precision":precision_score(y_test,test_pred,zero_division=0),
 "Sensitivity":recall_score(y_test,test_pred,zero_division=0),
 "Specificity":(confusion_matrix(y_test,test_pred).ravel()[0] /
                max(1,confusion_matrix(y_test,test_pred).ravel()[0]+confusion_matrix(y_test,test_pred).ravel()[1])),
 "F1-score":f1_score(y_test,test_pred,zero_division=0),
 "AUROC":roc_auc_score(y_test,test_prob) if len(np.unique(y_test))>1 else np.nan,
 "AUPRC":average_precision_score(y_test,test_prob),
 "Brier score":brier_score_loss(y_test,test_prob),
 "Expected Calibration Error":np.nan,
 "Referral rate":1-accepted.mean(),
 "Coverage":accepted.mean(),
 "Selective risk":selective_risk
}
def expected_calibration_error(y,p,n_bins=10):
    edges=np.linspace(0,1,n_bins+1); ece=0.
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(p>=lo)&(p<(hi if hi<1 else hi+1e-9))
        if mask.any(): ece += mask.mean()*abs(y[mask].mean()-p[mask].mean())
    return ece
metrics["Expected Calibration Error"]=expected_calibration_error(y_test,test_prob)
display(pd.DataFrame({"Metric":metrics.keys(),"Value":metrics.values()}))
print(classification_report(y_test,test_pred,target_names=["Benign","Malignant"],zero_division=0))
print("Decision counts:",pd.Series(test_decisions).value_counts().to_dict())

In [ ]:
cm=confusion_matrix(y_test,test_pred)
fig,ax=plt.subplots(figsize=(5,4))
im=ax.imshow(cm,cmap="Blues")
ax.set_xticks([0,1],["Benign","Malignant"]); ax.set_yticks([0,1],["Benign","Malignant"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Test confusion matrix")
for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha="center",va="center")
plt.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

frac_pos,mean_pred=calibration_curve(y_test,test_prob,n_bins=8,strategy="uniform")
plt.figure(figsize=(5,4)); plt.plot(mean_pred,frac_pos,"o-",label="PA-LUCDNet")
plt.plot([0,1],[0,1],"--",label="Perfect calibration")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed positive fraction")
plt.title("Calibration curve"); plt.legend(); plt.tight_layout(); plt.show()

## 10. Grad-CAM++ explanation for PACT and URCT branches

The implementation hooks the final convolutional feature map of each MobileNetV3-Small encoder and computes Grad-CAM++ weights for the malignant logit. It is an explanatory visualization, not proof of causal reasoning.

In [ ]:
def gradcam_plus_plus(model, image_tensor, modality="pact"):
    model.eval()
    encoder = model.pact_encoder if modality=="pact" else model.urct_encoder
    target_layer = encoder.features[-1]
    activations=[]; gradients=[]
    def fwd_hook(m,i,o): activations.append(o)
    def bwd_hook(m,gi,go): gradients.append(go[0])
    h1=target_layer.register_forward_hook(fwd_hook)
    h2=target_layer.register_full_backward_hook(bwd_hook)
    p=image_tensor.to(DEVICE)
    # For the unselected modality, use the same image only as a placeholder; CAM is branch-specific.
    model.zero_grad(set_to_none=True)
    if modality=="pact": logits=model(p,torch.zeros_like(p))
    else: logits=model(torch.zeros_like(p),p)
    score=logits.sum()
    score.backward()
    A=activations[0]; G=gradients[0]
    g2=G.pow(2); g3=G.pow(3)
    denom=2*g2 + (A*g3).sum((2,3),keepdim=True)
    alpha=g2/(denom+1e-8)
    weights=(alpha*F.relu(G)).sum((2,3),keepdim=True)
    cam=F.relu((weights*A).sum(1,keepdim=True))
    cam=F.interpolate(cam,size=p.shape[-2:],mode="bilinear",align_corners=False)
    cam=cam[0,0].detach().cpu().numpy()
    cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
    h1.remove(); h2.remove()
    return cam

# Visualize one test example
p,u,y,d,ph,ids=next(iter(test_loader))
cam_p=gradcam_plus_plus(model,p[:1],"pact")
cam_u=gradcam_plus_plus(model,u[:1],"urct")
fig,ax=plt.subplots(1,2,figsize=(9,4))
ax[0].imshow(p[0,0],cmap="gray"); ax[0].imshow(cam_p,cmap="jet",alpha=.4); ax[0].set_title("PACT Grad-CAM++"); ax[0].axis("off")
ax[1].imshow(u[0,0],cmap="gray"); ax[1].imshow(cam_u,cmap="jet",alpha=.4); ax[1].set_title("URCT Grad-CAM++"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 11. Ablation experiments

Run the same patient-level split for each setting. `PAFL off` removes MMD/physics penalties; `uncertainty off` uses one deterministic prediction and disables uncertainty referral. Compare all metrics on the same held-out patients. This cell provides the ablation configuration table; train each variant by setting the flags and rerunning the training/evaluation sections.

In [ ]:
ablation_plan = pd.DataFrame([
 ["MobileNetV3-Small + gated fusion",0,0,0],
 ["+ PAFL (MMD)",1,0,0],
 ["+ PAFL (MMD + physics)",1,1,0],
 ["Full PA-LUCDNet",1,1,1],
],columns=["Variant","MMD enabled","Physics loss enabled","MC-dropout/selective decision"])
display(ablation_plan)

## 12. INT8 quantization and efficiency measurement

The notebook attempts PyTorch FX static post-training quantization for CPU inference. Backend/operator support varies by PyTorch version; if conversion fails, the fallback dynamically quantizes supported linear layers (this is **not** full-network INT8). Always benchmark the resulting artifact on the target portable hardware.

In [ ]:
import copy
def model_size_mb(m):
    tmp=WORK_DIR/"_tmp_state.pt"
    torch.save(m.state_dict(),tmp)
    size=tmp.stat().st_size/1024**2
    tmp.unlink(missing_ok=True)
    return size

fp32_model=copy.deepcopy(model).cpu().eval()
quantized_model=None
quant_mode="not-created"
try:
    from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
    from torch.ao.quantization import get_default_qconfig_mapping
    example=(torch.randn(1,3,IMG_SIZE,IMG_SIZE),torch.randn(1,3,IMG_SIZE,IMG_SIZE))
    qconfig=get_default_qconfig_mapping("fbgemm")
    prepared=prepare_fx(copy.deepcopy(fp32_model),qconfig,example)
    # Calibration using representative training samples
    prepared.eval()
    with torch.no_grad():
        for batch in list(train_loader)[:20]:
            prepared(batch[0],batch[1])
    quantized_model=convert_fx(prepared)
    quant_mode="FX static quantization"
except Exception as e:
    print("FX static quantization unavailable:",repr(e))
    quantized_model=torch.ao.quantization.quantize_dynamic(
        copy.deepcopy(fp32_model),{nn.Linear},dtype=torch.qint8)
    quant_mode="dynamic INT8 Linear layers only (not full CNN INT8)"
print("Quantization mode:",quant_mode)
print("FP32 state size MB:",model_size_mb(fp32_model))
print("Quantized state size MB:",model_size_mb(quantized_model))

In [ ]:
def benchmark_inference(m, loader, max_batches=20):
    m.eval()
    times=[]
    with torch.inference_mode():
        for i,(p,u,y,d,ph,ids) in enumerate(loader):
            if i>=max_batches: break
            p,u=p.cpu(),u.cpu()
            t0=time.perf_counter(); _=m(p,u)
            times.append((time.perf_counter()-t0)*1000/len(p))
    return float(np.mean(times)),float(np.std(times))
fp_ms,fp_sd=benchmark_inference(fp32_model,test_loader)
q_ms,q_sd=benchmark_inference(quantized_model,test_loader)
eff=pd.DataFrame([
 ["FP32",sum(p.numel() for p in fp32_model.parameters()),model_size_mb(fp32_model),fp_ms,fp_sd],
 [quant_mode,sum(p.numel() for p in quantized_model.parameters()),model_size_mb(quantized_model),q_ms,q_sd]
],columns=["Model","Parameters","State size MB","Latency ms/image mean","Latency ms/image SD"])
display(eff)

## 13. Patient-level stratified cross-validation

This optional evaluation repeats training using patient-level stratified folds. It can be computationally expensive. The test partition above remains a final holdout; do not use it for hyperparameter selection. For publication, report fold-wise mean ± standard deviation and keep all paired modalities from a patient in the same fold.

In [ ]:
RUN_CV = False
N_FOLDS = 5
if RUN_CV:
    cv_rows=[]
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    for fold,(tr_idx,va_idx) in enumerate(skf.split(df,df.label),1):
        trf=df.iloc[tr_idx].reset_index(drop=True); vaf=df.iloc[va_idx].reset_index(drop=True)
        trl=DataLoader(PairedPACTDataset(trf,True),batch_size=BATCH_SIZE,shuffle=True,
                       num_workers=NUM_WORKERS,collate_fn=collate_batch)
        val=DataLoader(PairedPACTDataset(vaf),batch_size=BATCH_SIZE,shuffle=False,
                       num_workers=NUM_WORKERS,collate_fn=collate_batch)
        fold_model=PA_LUCDNet(pretrained=True).to(DEVICE)
        opt=torch.optim.AdamW(fold_model.parameters(),lr=LR,weight_decay=1e-4)
        for ep in range(EPOCHS):
            fold_model.train()
            for p,u,y,d,ph,ids in trl:
                p,u,y=p.to(DEVICE),u.to(DEVICE),y.to(DEVICE)
                opt.zero_grad(); loss=F.binary_cross_entropy_with_logits(fold_model(p,u),y)
                loss.backward(); opt.step()
        fold_model.eval(); probs=[]; labels=[]
        with torch.no_grad():
            for p,u,y,d,ph,ids in val:
                probs.extend(torch.sigmoid(fold_model(p.to(DEVICE),u.to(DEVICE))).cpu().numpy())
                labels.extend(y.numpy())
        pred=np.asarray(probs)>=.5
        cv_rows.append([fold,accuracy_score(labels,pred),precision_score(labels,pred,zero_division=0),
                        recall_score(labels,pred,zero_division=0),f1_score(labels,pred,zero_division=0),
                        roc_auc_score(labels,probs) if len(set(labels))>1 else np.nan])
        print("Fold",fold,cv_rows[-1][1:])
    cv_df=pd.DataFrame(cv_rows,columns=["Fold","Accuracy","Precision","Sensitivity","F1","AUROC"])
    display(cv_df)
    display(cv_df.drop(columns="Fold").agg(["mean","std"]))

## 14. Baseline comparisons

Use the same paired inputs, patient partitions, preprocessing, and training budget for MobileNetV3, EfficientNet-B0, and a conventional CNN. The helper below defines a single-modality baseline; for fair comparison, run it separately for PACT-only and URCT-only, then add a two-branch baseline without PAFL/uncertainty. Do not populate the comparison table with fabricated values.

In [ ]:
class SingleModalityBaseline(nn.Module):
    def __init__(self, backbone="mobilenet_v3_small", pretrained=True):
        super().__init__()
        if backbone=="mobilenet_v3_small":
            net=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None)
            in_features=net.classifier[-1].in_features
            net.classifier[-1]=nn.Linear(in_features,1)
        elif backbone=="efficientnet_b0":
            net=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
            in_features=net.classifier[-1].in_features
            net.classifier[-1]=nn.Linear(in_features,1)
        else:
            net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),nn.Linear(128,1))
        self.net=net
    def forward(self,x): return self.net(x).squeeze(1)

baseline_models=["mobilenet_v3_small","efficientnet_b0","conventional_cnn"]
print("Baseline architectures ready:",baseline_models)

## 15. Save model, metrics, predictions, and experiment metadata

In [ ]:
torch.save({
    "model_state_dict":model.state_dict(),
    "temperature":float(best_temp),
    "config":{"img_size":IMG_SIZE,"seed":SEED,"lambda_mmd":LAMBDA_MMD,
              "lambda_physics":LAMBDA_PHYSICS,"uncertainty_max":UNCERTAINTY_MAX,
              "low_risk_max":LOW_RISK_MAX,"high_risk_min":HIGH_RISK_MIN}
},WORK_DIR/"PA_LUCDNet_checkpoint.pth")

pred_df=pd.DataFrame({
    "patient_id":test_ids,"ground_truth":y_test,"malignancy_probability":test_prob,
    "predictive_uncertainty":test_unc,"predicted_class":test_pred,"screening_decision":test_decisions
})
pred_df.to_csv(WORK_DIR/"test_predictions.csv",index=False)
pd.DataFrame({"Metric":list(metrics.keys()),"Value":list(metrics.values())}).to_csv(WORK_DIR/"test_metrics.csv",index=False)
hist.to_csv(WORK_DIR/"training_history.csv",index=False)
with open(WORK_DIR/"experiment_config.json","w") as f:
    json.dump({"seed":SEED,"device":str(DEVICE),"image_size":IMG_SIZE,"epochs":EPOCHS,
               "batch_size":BATCH_SIZE,"temperature":float(best_temp),
               "quantization_mode":quant_mode},f,indent=2)
print("Saved artifacts in:",WORK_DIR.resolve())

## 16. Clinical and methodological safeguards

- The dataset is small (161 tumors); patient-level separation is essential, and performance uncertainty should be reported.
- The manifest must reflect verified patient/tumor IDs, paired modalities, and pathology labels.
- Domain adaptation is meaningful only when acquisition-domain labels are available and represented in training batches.
- Beer–Lambert loss requires calibrated optical measurements and consistent units. The default is disabled.
- Thresholds and uncertainty cutoffs shown here are starting configuration values, not validated clinical operating points.
- A model output is a research screening aid, not a diagnosis. Specialist review and prospective external validation are required before clinical deployment.